# 03 — PPO Agent: Training, Baseline Comparison, Pareto Frontier

Uses the refactored modules: `env.py` (environment), `agent.py` (networks +
training), `evaluate.py` (evaluation functions). No pasted code.

Trains four PPO configurations (the Pareto sweep), each saved as a model,
with convergence logging for the training-curve figures. Compares the best
config against the rule-based HPA baseline.

## 1. Imports and data

In [1]:
import json
import time
import numpy as np
import torch

from env import CloudClusterEnv, STEPS_PER_WEEK, MIN_PODS, MAX_PODS
from agent import ActorCritic, train_ppo
from evaluate import run_ppo, run_hpa

stats = json.load(open('trace_params.json'))['stats']
print("Imports ready. Steps per week:", STEPS_PER_WEEK)

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Imports ready. Steps per week: 672


## 2. Multi-episode evaluation helper

Runs an agent over several fresh episodes and averages the metrics.

In [2]:
def eval_ppo_avg(net, lambda_cost, lambda_sla, lambda_util, n_episodes=5):
    """Average a trained PPO agent over N episodes (fresh workloads)."""
    runs = []
    for _ in range(n_episodes):
        env = CloudClusterEnv(stats, lambda_cost=lambda_cost,
                              lambda_sla=lambda_sla, lambda_util=lambda_util)
        runs.append(run_ppo(env, net))
    return {
        'cost':     float(np.mean([r['cost'] for r in runs])),
        'breaches': float(np.mean([r['breaches'] for r in runs])),
        'util':     float(np.mean([r['util'] for r in runs])),
        'vms':      float(np.mean([r['vms'] for r in runs])),
    }

def eval_hpa_avg(n_episodes=5):
    """Average the HPA baseline over N episodes."""
    runs = []
    for _ in range(n_episodes):
        env = CloudClusterEnv(stats)
        runs.append(run_hpa(env))
    return {
        'cost':     float(np.mean([r['cost'] for r in runs])),
        'breaches': float(np.mean([r['breaches'] for r in runs])),
        'util':     float(np.mean([r['util'] for r in runs])),
        'vms':      float(np.mean([r['vms'] for r in runs])),
    }

print("Evaluation helpers ready.")

Evaluation helpers ready.


## 3. Pareto sweep — train four configurations

Trains four PPO agents spanning cost-focused to SLA-focused. Each model is
saved (`ppo_{name}.pth`), and each config's convergence curve is saved
(`ppo_{name}_convergence.json`) for the training-curve figures.

**This is the long-running cell (~1 hour: 4 configs × 250k steps).**

In [3]:
pareto_configs = [
    {'name': 'cost-focused',  'cost': 0.8, 'sla': 0.5, 'util': 0.1},
    {'name': 'balanced-cost', 'cost': 0.5, 'sla': 1.0, 'util': 0.1},
    {'name': 'balanced-sla',  'cost': 0.4, 'sla': 1.5, 'util': 0.1},
    {'name': 'sla-focused',   'cost': 0.3, 'sla': 2.5, 'util': 0.1},
]

pareto_results = []

for cfg in pareto_configs:
    print(f"\n{'='*55}")
    print(f"Training config: {cfg['name']} (cost={cfg['cost']}, sla={cfg['sla']})")
    print('='*55)

    # fresh env + network, train from scratch
    env = CloudClusterEnv(stats, lambda_cost=cfg['cost'],
                          lambda_sla=cfg['sla'], lambda_util=cfg['util'])
    net_cfg = ActorCritic()
    optimizer = torch.optim.Adam(net_cfg.parameters(), lr=3e-4)

    t0 = time.time()
    episode_rewards, convergence_log = train_ppo(env, net_cfg, optimizer,
                                                 total_steps=250_000)
    mins = (time.time() - t0) / 60

    # evaluate (5-episode average)
    metrics = eval_ppo_avg(net_cfg, cfg['cost'], cfg['sla'], cfg['util'], n_episodes=5)

    pareto_results.append({
        'name': cfg['name'],
        'lambda_cost': cfg['cost'], 'lambda_sla': cfg['sla'], 'lambda_util': cfg['util'],
        'cost': metrics['cost'], 'breaches': metrics['breaches'],
        'utilisation': metrics['util'], 'avg_vms': metrics['vms'],
    })

    # save model AND convergence log (for the training-curve figures)
    torch.save(net_cfg.state_dict(), f"ppo_{cfg['name']}.pth")
    json.dump(convergence_log, open(f"ppo_{cfg['name']}_convergence.json", 'w'))

    print(f"\n  → cost {metrics['cost']:.1f} | breaches {metrics['breaches']:.0f} "
          f"| util {metrics['util']:.3f} | VMs {metrics['vms']:.1f} "
          f"| {mins:.1f} min")

# save the collected pareto results
json.dump(pareto_results, open('pareto_results.json', 'w'), indent=2)

print("\n\nALL CONFIGS DONE. Saved models, convergence logs, pareto_results.json\n")
print(f"{'config':<16}{'cost':>8}{'breaches':>10}{'util':>8}{'avg_vms':>9}")
for r in pareto_results:
    print(f"{r['name']:<16}{r['cost']:>8.1f}{r['breaches']:>10.0f}"
          f"{r['utilisation']:>8.3f}{r['avg_vms']:>9.1f}")


Training config: cost-focused (cost=0.8, sla=0.5)
steps   2048 | recent ep reward   -302.1
steps   4096 | recent ep reward   -271.6
steps   6144 | recent ep reward   -247.6
steps   8192 | recent ep reward   -221.8
steps  10240 | recent ep reward   -177.2
steps  12288 | recent ep reward   -132.6
steps  14336 | recent ep reward   -121.1
steps  16384 | recent ep reward   -112.5
steps  18432 | recent ep reward   -103.5
steps  20480 | recent ep reward   -100.2
steps  22528 | recent ep reward   -100.3
steps  24576 | recent ep reward   -102.0
steps  26624 | recent ep reward   -100.6
steps  28672 | recent ep reward    -99.6
steps  30720 | recent ep reward   -101.0
steps  32768 | recent ep reward   -100.6
steps  34816 | recent ep reward   -100.3
steps  36864 | recent ep reward    -99.8
steps  38912 | recent ep reward    -98.5
steps  40960 | recent ep reward   -100.6
steps  43008 | recent ep reward    -99.4
steps  45056 | recent ep reward   -100.7
steps  47104 | recent ep reward   -103.8
steps 

## 4. Rule-based HPA baseline + head-to-head with best PPO

In [4]:
# measure the HPA baseline (5-episode average)
hpa = eval_hpa_avg(n_episodes=5)
print("Rule-based HPA baseline:")
print(f"  cost {hpa['cost']:.1f} | breaches {hpa['breaches']:.0f} "
      f"| util {hpa['util']:.3f} | VMs {hpa['vms']:.1f}\n")

# best config = sla-focused (loaded from its saved model)
best = ActorCritic()
best.load_state_dict(torch.load('ppo_sla-focused.pth'))
best.eval()
ppo_best = eval_ppo_avg(best, 0.3, 2.5, 0.1, n_episodes=5)

print("="*52)
print(f"{'METRIC':<16}{'HPA':>12}{'PPO (sla)':>13}")
print("="*52)
print(f"{'Total cost':<16}{hpa['cost']:>12.1f}{ppo_best['cost']:>13.1f}")
print(f"{'SLA breaches':<16}{hpa['breaches']:>12.0f}{ppo_best['breaches']:>13.0f}")
print(f"{'Utilisation':<16}{hpa['util']:>12.3f}{ppo_best['util']:>13.3f}")
print(f"{'Avg VMs':<16}{hpa['vms']:>12.1f}{ppo_best['vms']:>13.1f}")
print("="*52)
cost_d = (ppo_best['cost']-hpa['cost'])/hpa['cost']*100
breach_d = (ppo_best['breaches']-hpa['breaches'])/hpa['breaches']*100
print(f"\nPPO vs HPA:  cost {cost_d:+.1f}%,  breaches {breach_d:+.1f}%")

# save baseline for the figures notebook
json.dump({'hpa': hpa, 'ppo_best': ppo_best},
          open('baseline_comparison.json', 'w'), indent=2)
print("Saved baseline_comparison.json")

Rule-based HPA baseline:
  cost 292.4 | breaches 1845 | util 0.560 | VMs 8.7

METRIC                   HPA    PPO (sla)
Total cost             292.4        177.7
SLA breaches            1845          983
Utilisation            0.560        0.948
Avg VMs                  8.7          5.3

PPO vs HPA:  cost -39.2%,  breaches -46.7%
Saved baseline_comparison.json
